In [1]:
import pandas as pd
df = pd.read_csv("NGCC_Uncertainty_dataset.csv")

In [2]:
df

,Ambient Air Temperature (Dry Bulb Average) (°F),Natural Gas Cost ($/mscf),Cooling Water Temperature Rise (°F),Absorber CO2 Removal Efficiency (%),Amine Scrubber Power Requirement (% MWg),Important Performance and Cost Factors: CO2 Removed (tons/hr),Important Performance and Cost Factors: Net Electrical Output (MW),Costs of CO2 Avoided & Captured: Cost of CO2 Captured ($/ton),Makeup Water (tons/hr)
0,75.137497,6.725,33.900002,60.674999,4.4425,135.200363,491.935730,328.892700,0.000000
1,69.612503,5.875,43.700001,81.525002,5.9275,182.987564,477.473724,228.331604,0.000000
2,84.912498,11.325,33.299999,61.875000,1.6525,136.096603,499.090912,449.124939,0.000000
3,54.737499,2.425,15.300000,82.724998,5.5225,189.321732,489.426849,159.739182,789.642822
4,18.187500,8.875,36.099998,63.375000,1.7425,151.950363,547.840698,362.821564,625.265442
...,...,...,...,...,...,...,...,...,...
195,75.987503,9.025,11.700000,68.025002,2.8225,151.407898,496.516418,361.050995,882.480652
196,59.837502,9.875,32.700001,81.224998,6.3325,184.661026,482.363220,311.367065,199.811310
197,97.237503,9.725,19.100000,64.724998,5.4325,140.030426,469.309143,394.297119,0.000000
198,57.287498,3.975,12.700000,62.174999,9.3475,141.821533,477.319916,253.032211,772.738037


In [3]:
df.isna().sum()

Ambient Air Temperature (Dry Bulb Average) (°F)                       0
Natural Gas Cost ($/mscf)                                             0
Cooling Water Temperature Rise (°F)                                   0
Absorber CO2 Removal Efficiency (%)                                   0
Amine Scrubber Power Requirement (% MWg)                              0
Important Performance and Cost Factors: CO2 Removed (tons/hr)         0
Important Performance and Cost Factors: Net Electrical Output (MW)    0
Costs of CO2 Avoided & Captured: Cost of CO2 Captured ($/ton)         0
Makeup Water (tons/hr)                                                0
dtype: int64

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
x = df[['Ambient Air Temperature (Dry Bulb Average) (°F)',
    'Natural Gas Cost ($/mscf)',
    'Cooling Water Temperature Rise (°F)',
    'Absorber CO2 Removal Efficiency (%)',
    'Amine Scrubber Power Requirement (% MWg)']]

y = df['Important Performance and Cost Factors: CO2 Removed (tons/hr)']

In [6]:
x

,Ambient Air Temperature (Dry Bulb Average) (°F),Natural Gas Cost ($/mscf),Cooling Water Temperature Rise (°F),Absorber CO2 Removal Efficiency (%),Amine Scrubber Power Requirement (% MWg)
0,75.137497,6.725,33.900002,60.674999,4.4425
1,69.612503,5.875,43.700001,81.525002,5.9275
2,84.912498,11.325,33.299999,61.875000,1.6525
3,54.737499,2.425,15.300000,82.724998,5.5225
4,18.187500,8.875,36.099998,63.375000,1.7425
...,...,...,...,...,...
195,75.987503,9.025,11.700000,68.025002,2.8225
196,59.837502,9.875,32.700001,81.224998,6.3325
197,97.237503,9.725,19.100000,64.724998,5.4325
198,57.287498,3.975,12.700000,62.174999,9.3475


In [7]:
y

0      135.200363
1      182.987564
2      136.096603
3      189.321732
4      151.950363
          ...    
195    151.407898
196    184.661026
197    140.030426
198    141.821533
199    168.934784
Name: Important Performance and Cost Factors: CO2 Removed (tons/hr), Length: 200, dtype: float64

In [8]:
from sklearn.model_selection import GridSearchCV, train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("\n=== REGRESSION MODELS ===\n")


=== REGRESSION MODELS ===



In [9]:
# Linear Regression
reg_pipe = Pipeline([
    ('scaler',StandardScaler()),
    ('reg',LinearRegression())
])
reg_grid = GridSearchCV(reg_pipe, {}, cv=5, scoring='r2', n_jobs=-1)
reg_grid.fit(x_train,y_train)
print("Linear Regression best params:",reg_grid.best_params_)

best_reg = reg_grid.best_estimator_
print('Train R2:',best_reg.score(x_train,y_train))
print('Test R2:',best_reg.score(x_test,y_test))

Linear Regression best params: {}
Train R2: 0.9991435492047871
Test R2: 0.9986859349666222


In [10]:
# SVR
svr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR())
])
svr_grid = GridSearchCV(
    svr_pipe,
    {
        "svr__kernel": ["linear", "rbf"],
        "svr__C": [0.01, 0.1, 1, 10,100],
        "svr__gamma": ["scale"]
    },
    cv=5, scoring="r2", n_jobs=-1
)
svr_grid.fit(x_train, y_train)

best_svr = svr_grid.best_estimator_
print("SVR best params:", svr_grid.best_params_)
print('Train R2:',best_svr.score(x_train,y_train))
print('Test R2:',best_svr.score(x_test,y_test))

SVR best params: {'svr__C': 100, 'svr__gamma': 'scale', 'svr__kernel': 'linear'}
Train R2: 0.9991345525359542
Test R2: 0.9986636359877288


In [11]:
# XGBoost Regressor
xgb_grid = GridSearchCV(
    XGBRegressor(random_state=42),
    {
        "n_estimators": [100, 300],
        "max_depth": [3, 6, 10],
        "learning_rate": [0.01, 0.1, 0.3],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0]
    },
    cv=5, scoring="r2", n_jobs=-1
)
xgb_grid.fit(x_train, y_train)

best_xgb = xgb_grid.best_estimator_
print("XGBoost best params:", xgb_grid.best_params_)
print('Train R2:',best_xgb.score(x_train,y_train))
print('Test R2:',best_xgb.score(x_test,y_test))

XGBoost best params: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.8}
Train R2: 0.999982362561815
Test R2: 0.9899827747041059


In [12]:
# Gradient Boosting Regressor
gbr_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    {
        "n_estimators": [100, 300],
        "max_depth": [3, 6, 10],
        "learning_rate": [0.01, 0.1, 0.3],
        "subsample": [0.8, 1.0],
        "max_features": [0.8, 1.0]
    },
    cv=5, scoring="r2", n_jobs=-1
)
gbr_grid.fit(x_train, y_train)

best_gbr = gbr_grid.best_estimator_
print("Gradient Boosting best params:", gbr_grid.best_params_)
print('Train R2:', best_gbr.score(x_train, y_train))
print('Test R2:', best_gbr.score(x_test, y_test))

Gradient Boosting best params: {'learning_rate': 0.1, 'max_depth': 3, 'max_features': 0.8, 'n_estimators': 300, 'subsample': 0.8}
Train R2: 0.999988987082669
Test R2: 0.9951071238326308


In [13]:
# Random Forest Regressor
rfr_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    {
        "n_estimators": [100,300],
        "max_depth": [None, 5, 10, 20],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ['sqrt',0.5, 0.7]
    },
    cv=5, scoring="r2", n_jobs=-1
)
rfr_grid.fit(x_train, y_train)

best_rfr = rfr_grid.best_estimator_
print("RF best params:", rfr_grid.best_params_)
print('Train R2:',best_rfr.score(x_train,y_train))
print('Test R2:',best_rfr.score(x_test,y_test))

RF best params: {'max_depth': 10, 'max_features': 0.7, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Train R2: 0.9975155100962646
Test R2: 0.9841599009244391


In [14]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge

poly_pipe = Pipeline([
    ('poly',   PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg',    Ridge())
])
poly_grid = GridSearchCV(
    poly_pipe,
    {'poly__degree': [2, 3, 4],
     'reg__alpha':  [0.001, 0.01, 0.1, 1, 10, 100]},
    cv=5, scoring='r2', n_jobs=-1
)
poly_grid.fit(x_train, y_train)

best_poly = poly_grid.best_estimator_
print("Polynomial best params:", poly_grid.best_params_)
print('Train R2:', best_poly.score(x_train, y_train))
print('Test R2:', best_poly.score(x_test, y_test))

Polynomial best params: {'poly__degree': 2, 'reg__alpha': 0.001}
Train R2: 0.9999999698895434
Test R2: 0.9999999475041553


In [15]:
#Checking the RMSE and MAPE scores
from sklearn.metrics import mean_squared_error
import numpy as np

print("\n=== RMSE & MAPE ===\n")

models = {
    'Linear Regression': best_reg,
    'SVR':               best_svr,
    'Random Forest':     best_rfr,
    'XGBoost':           best_xgb,
    'Gradient Boost':    best_gbr,
    'Polynomial':        best_poly
}

for name, model in models.items():
    y_pred = model.predict(x_test)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    mape   = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    print(f"{name} | RMSE: {rmse:.4f} | MAPE: {mape:.4f}%")


=== RMSE & MAPE ===

Linear Regression | RMSE: 0.7291 | MAPE: 0.3354%
SVR | RMSE: 0.7352 | MAPE: 0.3349%
Random Forest | RMSE: 2.5312 | MAPE: 1.2139%
XGBoost | RMSE: 2.0129 | MAPE: 0.9104%
Gradient Boost | RMSE: 1.4068 | MAPE: 0.6525%
Polynomial | RMSE: 0.0046 | MAPE: 0.0020%


In [16]:
#Exporting to excel
import os
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

file       = 'NGCC_CO2_Removed.xlsx'
experiment = 'Target_1_NGCC_CO2_Removed'   # change per experiment

results = []

models = {
    'Linear Regression': best_reg,
    'SVR':               best_svr,
    'Random Forest':     best_rfr,
    'XGBoost':           best_xgb,
    'Gradient Boost':    best_gbr,
    'Polynomial':        best_poly
}

for name, model in models.items():
    y_pred = model.predict(x_test)
    results.append({
        'Experiment': experiment,
        'Model':      name,
        'Train R2':   round(model.score(x_train, y_train), 4),
        'Test R2':    round(model.score(x_test, y_test), 4),
        'MAE':        round(mean_absolute_error(y_test, y_pred), 4),
        'MSE':        round(mean_squared_error(y_test, y_pred), 4),
        'RMSE':       round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
        'MAPE (%)':   round(np.mean(np.abs((y_test - y_pred) / y_test)) * 100, 4)
    })

results_df = pd.DataFrame(results)

if os.path.exists(file):
    existing = pd.read_excel(file)
    combined = pd.concat([existing, results_df], ignore_index=True)
    combined.to_excel(file, index=False)
else:
    results_df.to_excel(file, index=False)

print(results_df)

                  Experiment              Model  Train R2  Test R2     MAE  \
0  Target_1_NGCC_CO2_Removed  Linear Regression    0.9991   0.9987  0.5638   
1  Target_1_NGCC_CO2_Removed                SVR    0.9991   0.9987  0.5675   
2  Target_1_NGCC_CO2_Removed      Random Forest    0.9975   0.9842  2.0209   
3  Target_1_NGCC_CO2_Removed            XGBoost    1.0000   0.9900  1.5777   
4  Target_1_NGCC_CO2_Removed     Gradient Boost    1.0000   0.9951  1.0962   
5  Target_1_NGCC_CO2_Removed         Polynomial    1.0000   1.0000  0.0034   

      MSE    RMSE  MAPE (%)  
0  0.5315  0.7291    0.3354  
1  0.5405  0.7352    0.3349  
2  6.4071  2.5312    1.2139  
3  4.0519  2.0129    0.9104  
4  1.9791  1.4068    0.6525  
5  0.0000  0.0046    0.0020  
